### 파생 변수 생성

In [1]:
import pandas as pd
import numpy as np

c:\Users\dddhs\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\dddhs\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [14]:
# =========================
# 1. 데이터 불러오기
# =========================
data_2008 = pd.read_csv(
    r'C:..\..\data\raw\data_2008.csv',
    index_col=0,
    parse_dates=True
)

data_2009 = pd.read_csv(
    r'C:..\..\data\raw\data_2009.csv',
    index_col=0,
    parse_dates=True
)


In [15]:
# 인덱스 정렬
data_2008 = data_2008.sort_index()
data_2009 = data_2009.sort_index()

# =========================
# 2. 기준 컬럼
# =========================
price_col = 'KOSPI 200 Close'

# =========================
# 3. MA / EMA 생성
# =========================
data_2008['KOSPI 200_MA5'] = data_2008[price_col].rolling(window=5).mean()
data_2008['KOSPI 200_MA15'] = data_2008[price_col].rolling(window=15).mean()

data_2008['KOSPI 200_EMA5'] = data_2008[price_col].ewm(span=5, adjust=False).mean()
data_2008['KOSPI 200_EMA15'] = data_2008[price_col].ewm(span=15, adjust=False).mean()


In [16]:
# =========================
# 4. RSI(14) 생성
# =========================
delta = data_2008[price_col].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

# 단순이동평균 방식 RSI
avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss
data_2008['KOSPI 200_RSI14'] = 100 - (100 / (1 + rs))

# 만약 loss가 0이면 RSI가 100에 가까워질 수 있음
# 필요시 아래처럼 무한대 처리 가능
data_2008['KOSPI 200_RSI14'] = data_2008['KOSPI 200_RSI14'].replace([np.inf, -np.inf], np.nan)


In [17]:
# =========================
# 5. Bollinger Bands(15, 2) 생성
# =========================
bb_mid = data_2008[price_col].rolling(window=15).mean()
bb_std = data_2008[price_col].rolling(window=15).std()

data_2008['KOSPI 200_BB_MID20'] = bb_mid
data_2008['KOSPI 200_BB_UPPER20'] = bb_mid + 2 * bb_std
data_2008['KOSPI 200_BB_LOWER20'] = bb_mid - 2 * bb_std

# =========================

In [18]:
# =========================
# 6. 2009 데이터에 붙일 피처 목록
# =========================
feature_cols = [
    'KOSPI 200_MA5',
    'KOSPI 200_MA15',
    'KOSPI 200_EMA5',
    'KOSPI 200_EMA15',
    'KOSPI 200_RSI14',
    'KOSPI 200_BB_MID20',
    'KOSPI 200_BB_UPPER20',
    'KOSPI 200_BB_LOWER20'
]

# =========================
# 7. data_2009 인덱스 기준으로 결합
# =========================
data_2009[feature_cols] = data_2008[feature_cols].reindex(data_2009.index)

# 백업
data_tech_features = data_2009.copy()

In [19]:
# =========================
# 8. 확인
# =========================
print(data_tech_features.head())
print("\n결측치 개수:")
print(data_tech_features[feature_cols].isna().sum())

# =========================

            Shanghai Comp  KODEX 200  TOPIX  Brent Crude Oil  USD/CNY  \
Date                                                                    
2009-04-17    2503.935059    17370.0  875.0        51.959999   6.8226   
2009-04-20    2557.456055    17480.0  876.0        51.959999   6.8230   
2009-04-21    2535.827881    17480.0  855.0        51.959999   6.8169   
2009-04-22    2461.345947    17715.0  856.0        51.959999   6.8195   
2009-04-23    2463.954102    17895.0  862.0        51.959999   6.8193   

             Gold Spot  KRW/JPY      KRW/USD       NASDAQ      KOSDAQ  ...  \
Date                                                                   ...   
2009-04-17  867.400024   13.371  1325.800049  1673.069946  483.799988  ...   
2009-04-20  887.000000   13.536  1327.500000  1608.209961  491.940002  ...   
2009-04-21  882.099976   13.727  1354.300049  1643.849976  497.190002  ...   
2009-04-22  891.799988   13.726  1346.599976  1646.119995  509.899994  ...   
2009-04-23  905.9000

In [20]:
#data_tech_features 데이터의 USD/CNY 환율 컬럼을 삭제
data_tech_features = data_tech_features.drop(columns=['USD/CNY'])

In [21]:
data_tech_features.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 4109 entries, 2009-04-17 to 2026-02-27
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Shanghai Comp         4109 non-null   float64
 1   KODEX 200             4109 non-null   float64
 2   TOPIX                 4109 non-null   float64
 3   Brent Crude Oil       4109 non-null   float64
 4   Gold Spot             4109 non-null   float64
 5   KRW/JPY               4109 non-null   float64
 6   KRW/USD               4109 non-null   float64
 7   NASDAQ                4109 non-null   float64
 8   KOSDAQ                4109 non-null   float64
 9   KOSPI 200 Close       4109 non-null   float64
 10  KOSPI 200 Open        4109 non-null   float64
 11  KOSPI 200 High        4109 non-null   float64
 12  KOSPI 200 Low         4109 non-null   float64
 13  KRW/CNY               4109 non-null   float64
 14  KOSPI 200_MA5         4109 non-null   float64
 15  KOSPI 200_MA15

In [22]:
data_tech_features.head(10)

,Shanghai Comp,KODEX 200,TOPIX,Brent Crude Oil,Gold Spot,KRW/JPY,KRW/USD,NASDAQ,KOSDAQ,KOSPI 200 Close,...,KOSPI 200 Low,KRW/CNY,KOSPI 200_MA5,KOSPI 200_MA15,KOSPI 200_EMA5,KOSPI 200_EMA15,KOSPI 200_RSI14,KOSPI 200_BB_MID20,KOSPI 200_BB_UPPER20,KOSPI 200_BB_LOWER20
Date,,,,,,,,,,,,,,,,,,,,,
2009-04-17,2503.935059,17370.0,875.0,51.959999,867.400024,13.371,1325.800049,1673.069946,483.799988,171.330002,...,169.710007,194.324755,171.598001,166.383999,170.855038,166.505376,63.873746,166.383999,177.305127,155.462870
2009-04-20,2557.456055,17480.0,876.0,51.959999,887.000000,13.536,1327.500000,1608.209961,491.940002,172.300003,...,169.039993,194.562510,171.720001,167.093332,171.336693,167.229705,76.439644,167.093332,178.081434,156.105230
2009-04-21,2535.827881,17480.0,855.0,51.959999,882.099976,13.727,1354.300049,1643.849976,497.190002,171.960007,...,167.660004,198.668030,171.706003,168.141999,171.544464,167.820992,74.958299,168.141999,177.579936,158.704063
2009-04-22,2461.345947,17715.0,856.0,51.959999,891.799988,13.726,1346.599976,1646.119995,509.899994,174.399994,...,171.860001,197.463154,172.344000,169.301333,172.496308,168.643368,74.084326,169.301333,176.988826,161.613840
2009-04-23,2463.954102,17895.0,862.0,51.959999,905.900024,13.618,1333.599976,1652.209961,514.090027,176.139999,...,173.899994,195.562586,173.226001,170.346665,173.710872,169.580447,69.951956,170.346665,177.087774,163.605556
2009-04-24,2448.594971,17690.0,858.0,51.959999,913.599976,13.775,1340.300049,1694.290039,507.500000,174.130005,...,172.750000,196.614310,173.786002,170.876666,173.850583,170.149141,63.479290,170.876666,177.462103,164.291230
2009-04-27,2405.347900,17495.0,855.0,51.959999,907.400024,13.989,1350.500000,1679.410034,505.970001,172.130005,...,171.199997,198.130930,173.752002,171.211333,173.277057,170.396749,56.615179,171.211333,177.478959,164.943707
2009-04-28,2401.437988,17025.0,839.0,51.959999,892.799988,13.927,1344.900024,1673.810059,479.369995,167.240005,...,167.139999,197.268844,172.808002,171.119334,171.264706,170.002156,48.233796,171.119334,177.587286,164.651382
2009-04-29,2468.191895,17325.0,839.0,51.959999,899.799988,13.752,1341.300049,1711.939941,494.470001,172.080002,...,167.279999,196.815853,172.344003,171.368668,171.536471,170.261887,65.501354,171.368668,177.663474,165.073861


In [23]:
data_tech_features.to_csv(
    r'..\..\data\processed\data_tech_features.csv',
    index=True
)